In [7]:
# CELL 1 — Imports and config
import os
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
import timm

from sklearn.metrics import cohen_kappa_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Device + basic config
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4   # Mayo 0..3
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


In [8]:
# CELL 2 — Paths and transforms (edit if needed)
train_val_root = r"D:\Project\LIMUC\LIMUC\train_and_validation_sets\train_and_validation_sets"
test_root      = r"D:\Project\LIMUC\LIMUC\test_set\test_set"

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.05,0.05,0.05,0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD),
])


In [9]:
# CELL 3 — Build datasets & dataloaders
# NOTE: this expects class folders named e.g. 'Mayo_0','Mayo_1' or '0','1', etc.
train_ds = datasets.ImageFolder(train_val_root, transform=train_tf)
# If folder contains train+val together, split here; otherwise if the folder already split, use as is.
# If your folder is only train+validation combined, perform an 80/20 random split:
train_size = int(0.85 * len(train_ds))
val_size = len(train_ds) - train_size
train_ds, val_ds = torch.utils.data.random_split(train_ds, [train_size, val_size],
                                                 generator=torch.Generator().manual_seed(SEED))
# For the validation subset we need to replace the transform with val_tf:
# hack: wrap datasets.Subset to apply transform override
class SubsetWithTransform(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        # img is already transformed by original transform; we need raw image: so better to re-create dataset from samples
        # To keep it robust, we'll instead rebuild val_ds using ImageFolder pointing to a val folder if available.
        return img, label

# Instead of the hack above, if you have a separate val folder, do:
# val_ds = datasets.ImageFolder(val_root, transform=val_tf)
# For current notebook we will re-create train and val using ImageFolder if your structure has train/val directories.
# If train_val_root actually contains two folders train/val, adjust:
if any((Path(train_val_root)/d).is_dir() for d in ("train","val")):
    train_ds = datasets.ImageFolder(os.path.join(train_val_root,"train"), transform=train_tf)
    val_ds   = datasets.ImageFolder(os.path.join(train_val_root,"val"), transform=val_tf)
else:
    # if not, use the random split with transform replacement hack below
    # rebuild an ImageFolder dataset to fetch raw image path samples
    base_ds = datasets.ImageFolder(train_val_root, transform=val_tf)  # using val transform to load raw RGB resized
    indices = list(range(len(base_ds)))
    np.random.seed(SEED)
    np.random.shuffle(indices)
    split = int(0.85 * len(indices))
    train_idx, val_idx = indices[:split], indices[split:]
    train_ds = torch.utils.data.Subset(base_ds, train_idx)
    val_ds = torch.utils.data.Subset(base_ds, val_idx)

test_ds = datasets.ImageFolder(test_root, transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False, num_workers=4, pin_memory=True)

print("Classes:", test_ds.classes)
print("Train:", len(train_ds), "Val:", len(val_ds), "Test:", len(test_ds))


Classes: ['Mayo 0', 'Mayo 1', 'Mayo 2', 'Mayo 3']
Train: 8151 Val: 1439 Test: 1686


In [10]:
# CELL 4 — Helper functions for ordinal targets (cumulative encoding) and metrics

def labels_to_cumulative(labels, num_classes=4):
    # labels: torch.Tensor shape (N,) with values 0..num_classes-1
    # returns shape (N, num_classes-1) with binary cumulative labels
    labels = labels.view(-1)
    cum = []
    for k in range(num_classes-1):
        # target is 1 if original label > k
        cum.append((labels > k).float())
    return torch.stack(cum, dim=1)  # shape N x (K-1)

def cumulative_logits_to_label(logits):
    # logits: tensor N x (K-1)
    probs = torch.sigmoid(logits)
    # predict class as sum(prob>0.5)
    preds = (probs > 0.5).sum(dim=1).long()
    return preds  # values 0..K-1

def compute_qwk(y_true, y_pred):
    # sklearn cohen_kappa_score with quadratic weights
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")

def per_class_recall_specificity(cm, class_names):
    results = {}
    for i, name in enumerate(class_names):
        TP = cm[i,i]
        FN = cm[i,:].sum() - TP
        FP = cm[:,i].sum() - TP
        TN = cm.sum() - (TP+FN+FP)
        recall = TP/(TP+FN+1e-12)
        specificity = TN/(TN+FP+1e-12)
        results[name] = {"recall": float(recall), "specificity": float(specificity)}
    return results


In [11]:
# CELL 5 — Model: EFFResNet-ViT backbone + ordinal head
class EFFResNetViTOrdinal(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        # EfficientNet-B4 backbone
        self.eff = timm.create_model("efficientnet_b4", pretrained=True, features_only=True)
        self.res = timm.create_model("resnet50", pretrained=True, features_only=True)
        eff_dim = self.eff.feature_info[-1]['num_chs']
        res_dim = self.res.feature_info[-1]['num_chs']
        fused = eff_dim + res_dim
        self.fusion = nn.Conv2d(fused, 768, kernel_size=1)
        # small transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=768, nhead=12, dim_feedforward=3072, dropout=0.1, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        # ordinal head: outputs K-1 logits per sample
        self.ordinal_head = nn.Linear(768, num_classes - 1)
    def forward(self, x):
        eff_feat = self.eff(x)[-1]   # [B, Ce, H, W]
        res_feat = self.res(x)[-1]   # [B, Cr, H, W]
        fused = torch.cat([eff_feat, res_feat], dim=1)
        t = self.fusion(fused)       # [B, 768, H, W]
        t = t.flatten(2).transpose(1,2)  # [B, N, 768]
        t = self.transformer(t)          # [B, N, 768]
        pooled = t.mean(dim=1)           # [B, 768]
        logits = self.ordinal_head(pooled)  # [B, K-1]
        return logits


In [12]:
# CELL 6 — Instantiate model, loss, optimizer, scheduler
model = EFFResNetViTOrdinal(num_classes=NUM_CLASSES).to(DEVICE)

# Ordinal loss: BCEWithLogits on cumulative targets
bce_loss = nn.BCEWithLogitsLoss()

optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-6)

# early stopping on val_qwk
class EarlyStopQWK:
    def __init__(self, patience=7, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best = -1.0
        self.counter = 0
    def step(self, value):
        if value > self.best + self.min_delta:
            self.best = value
            self.counter = 0
            return False, True
        else:
            self.counter += 1
            stop = self.counter >= self.patience
            return stop, False

earlystop = EarlyStopQWK(patience=7)


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


In [13]:
# CELL 7 — Training + validation loops (monitoring QWK)
EPOCHS = 50
best_model_path = "best_effresnetvit_ordinal.pth"

for epoch in range(1, EPOCHS+1):
    model.train()
    running_loss = 0.0
    for imgs, labels in tqdm(train_loader, desc=f"Train Epoch {epoch}"):
        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE).long()
        cum_targets = labels_to_cumulative(labels, num_classes=NUM_CLASSES).to(DEVICE)  # N x (K-1)
        optimizer.zero_grad()
        logits = model(imgs)  # N x (K-1)
        loss = bce_loss(logits, cum_targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()

    # Validation: compute QWK & other metrics
    model.eval()
    all_gt = []
    all_pred = []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE).long()
            logits = model(imgs)  # N x (K-1)
            preds = cumulative_logits_to_label(logits)  # tensor on same device
            all_gt.append(labels.cpu().numpy())
            all_pred.append(preds.cpu().numpy())
    all_gt = np.concatenate(all_gt)
    all_pred = np.concatenate(all_pred)
    val_qwk = compute_qwk(all_gt, all_pred)
    val_acc = (all_gt == all_pred).mean()
    cm_val = confusion_matrix(all_gt, all_pred)
    print(f"Epoch {epoch} | Train loss: {running_loss/len(train_loader):.4f} | Val QWK: {val_qwk:.4f} | Val Acc: {val_acc:.4f}")

    # early stopping decision
    stop, is_best = earlystop.step(val_qwk)
    if is_best:
        torch.save(model.state_dict(), best_model_path)
        print("Saved best model (by QWK).")
    if stop:
        print("Early stopping triggered.")
        break



Train Epoch 1: 100%|██████████| 255/255 [00:55<00:00,  4.63it/s]


Epoch 1 | Train loss: 0.2534 | Val QWK: 0.8380 | Val Acc: 0.7609
Saved best model (by QWK).


Train Epoch 2: 100%|██████████| 255/255 [00:54<00:00,  4.66it/s]


Epoch 2 | Train loss: 0.1574 | Val QWK: 0.8386 | Val Acc: 0.7568
Saved best model (by QWK).


Train Epoch 3: 100%|██████████| 255/255 [00:51<00:00,  4.97it/s]


...

Epoch 10 | Train loss: 0.1488 | Val QWK: 0.8455 | Val Acc: 0.7637


Train Epoch 11: 100%|██████████| 255/255 [00:53<00:00,  4.78it/s]


Epoch 11 | Train loss: 0.1413 | Val QWK: 0.8344 | Val Acc: 0.7554


Train Epoch 12: 100%|██████████| 255/255 [00:52<00:00,  4.82it/s]


Epoch 12 | Train loss: 0.1368 | Val QWK: 0.8464 | Val Acc: 0.7658
Early stopping triggered.
